# Telecom Customer Churn Prediction - Logistic Regression

This notebook trains and evaluates a Logistic Regression model for predicting customer churn.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for plots
plt.style.use('ggplot')

## 1. Data Loading

In [ ]:
# Load the dataset
data_path = '../Dataset/Processed_Telecom_Churn.csv'
df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
df.head()

## 2. Data Preparation
Here we will split the dataset into features (X) and the target variable (y), then segment it into training and testing portions. Finally, we apply feature scaling.

In [ ]:
# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Split dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set shape:", X_train_scaled.shape)
print("Testing set shape:", X_test_scaled.shape)

## 3. Model Training
We initialize our `LogisticRegression` class and fit it over the training sets.

In [ ]:
# Initialize and train the Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

print("Model trained successfully!")

## 4. Model Evaluation
In this section, we predict testing sets and overview metrics like Accuracy, Precision, Recall, F1 Score, ROC-AUC and visual confusion matrix.

In [ ]:
# Predict on the test set
y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

# Evaluation metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Churn', 'Churn'], 
            yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 5. Export Predictions
Here we export the final predictions into a CSV file, similar to how the processed dataset was saved.

In [ ]:
# Export predictions to CSV file
predictions_df = X_test.copy()
predictions_df['Actual_Churn'] = y_test
predictions_df['Predicted_Churn'] = y_pred
predictions_df['Predicted_Probability'] = np.round(y_pred_proba, 4)

# Create Results directory if it doesn't exist just in case
if not os.path.exists('../Results'):
    os.makedirs('../Results')

save_path = '../Results/Logistic_Regression_Predictions.csv'
predictions_df.to_csv(save_path, index=False)
print(f"Successfully saved predictions to: {save_path}")